# BARAM 2026 — MLP 앙상블 멤버 실험 (비선형 다양성)

**목표**: MLP 단독으로 GBM을 이기는 게 아니라, **오차 방향이 다른 앙상블 멤버**로 공식 총점을 올릴 수 있는지 검증.
근거: KDD Cup 2022 상위권 = GBDT+NN 앙상블. 현 앙상블(LGBM+HistGBM)은 둘 다 트리라 다양성 부족(이득 −0.05%p뿐).

**설계**
- pooled MLP: 3그룹 합침 + 그룹 임베딩 → 학습 행 ~7만(소량 완화). feature = v2 65개 표준화, 타깃 = CF(0~1), 손실 = L1(지표 정합).
- 65→128→128→1, GELU, dropout 0.1, weight decay, 내부 시간순 뒤 15% early stopping.
- CV(expanding): GBM_ENS vs MLP 단독 vs blend(0.75/0.25, 0.5/0.5) — 공식 총점.
- 채택 기준: blend가 GBM 단독을 **2023·2024 두 해 모두** 이길 때만.

> torch+lightgbm OpenMP 충돌 방지: 첫 셀 OMP 가드 필수.

## 0. 설정

In [1]:
import os
os.environ.setdefault("OMP_NUM_THREADS","1")
import warnings; warnings.filterwarnings("ignore")
import json, numpy as np, pandas as pd
import torch, torch.nn as nn
torch.set_num_threads(1); torch.manual_seed(0); np.random.seed(0)
import lightgbm as lgb
from sklearn.ensemble import HistGradientBoostingRegressor as HGB
import wind_lib as W
from official_metric import group_scores

GROUPS=(1,2,3); FR={}; TGT={}
for g in GROUPS:
    df,tgt=W.load_train(g); TGT[g]=tgt
    FR[g]=W.add_spatial(W.build(df,g),"train")
BASE_ALL=[c for c in W.feature_cols(FR[1]) if c not in W.SPATIAL_COLS]+["pc_pred_cf"]
V2=W.lean_features(BASE_ALL)+W.SPATIAL_COLS
print("V2 features:", len(V2))

def lgbm(): return lgb.LGBMRegressor(objective="mae",n_estimators=600,learning_rate=0.03,num_leaves=63,
    min_child_samples=60,subsample=0.8,subsample_freq=1,colsample_bytree=0.7,reg_lambda=1.0,
    random_state=42,n_jobs=1,verbose=-1)
def histgbm(): return HGB(loss="absolute_error",max_iter=600,learning_rate=0.03,max_leaf_nodes=63,
    l2_regularization=1.0,random_state=42)
def gbm_ens(tr,va,cols,tgt):
    lg=lgbm().fit(tr[cols],tr[tgt]); hg=histgbm().fit(tr[cols].to_numpy(),tr[tgt].to_numpy())
    return 0.5*(lg.predict(va[cols])+hg.predict(va[cols].to_numpy()))

V2 features: 65


## 1. MLP 정의 & 학습 루틴

In [2]:
class MLP(nn.Module):
    def __init__(s, nf, ng=3, h=128, emb=4):
        super().__init__()
        s.emb=nn.Embedding(ng,emb)
        s.net=nn.Sequential(nn.Linear(nf+emb,h), nn.GELU(), nn.Dropout(0.1),
                            nn.Linear(h,h), nn.GELU(), nn.Dropout(0.1), nn.Linear(h,1))
    def forward(s,x,g):
        return s.net(torch.cat([x,s.emb(g)],-1)).squeeze(-1)

def train_mlp(pool_tr, feats, epochs=120, bs=512, lr=1e-3):
    """pool_tr: columns feats + ['cf','gid','kst_dtm']. 시간순 뒤 15% early stop."""
    pool_tr=pool_tr.sort_values("kst_dtm")
    mu=pool_tr[feats].mean(); sd=pool_tr[feats].std()+1e-8
    X=((pool_tr[feats]-mu)/sd).to_numpy(np.float32)
    y=pool_tr["cf"].to_numpy(np.float32); gid=pool_tr["gid"].to_numpy(np.int64)
    n=len(X); cut=int(n*0.85)
    Xt=torch.tensor(X); yt=torch.tensor(y); gt=torch.tensor(gid)
    net=MLP(len(feats)); opt=torch.optim.AdamW(net.parameters(),lr=lr,weight_decay=1e-4)
    best=1e9; bs_state=None; pat=0
    tr_idx=np.arange(cut); va_idx=np.arange(cut,n)
    for ep in range(epochs):
        net.train(); perm=np.random.permutation(tr_idx)
        for i in range(0,len(perm),bs):
            b=perm[i:i+bs]; opt.zero_grad()
            loss=(net(Xt[b],gt[b])-yt[b]).abs().mean(); loss.backward(); opt.step()
        net.eval()
        with torch.no_grad(): vl=(net(Xt[va_idx],gt[va_idx])-yt[va_idx]).abs().mean().item()
        if vl<best-1e-5: best=vl; bs_state={k:v.clone() for k,v in net.state_dict().items()}; pat=0
        else: pat+=1
        if pat>=10: break
    net.load_state_dict(bs_state)
    return net,(mu,sd),best,ep

def mlp_predict(net, scaler, fr, feats, g, cap):
    mu,sd=scaler
    X=torch.tensor(((fr[feats]-mu)/sd).to_numpy(np.float32))
    gid=torch.full((len(fr),),g-1,dtype=torch.long)
    net.eval()
    with torch.no_grad(): p=net(X,gid).numpy()
    return np.clip(p,0,1)*cap

## 2. Expanding CV — GBM vs MLP vs blends

In [3]:
FOLDS={1:[([2022],2023),([2022,2023],2024)],2:[([2022],2023),([2022,2023],2024)],3:[([2023],2024)]}
YEAR_FOLDS=[([2022],2023),([2022,2023],2024)]

rows=[]
for tys,vy in YEAR_FOLDS:
    # 이 폴드에서 학습 가능한 그룹 데이터 pool
    pool=[]; gbm_pred={}; va_frames={}
    for g in GROUPS:
        tgt=TGT[g]; cap=W.CAP[g]; fr=FR[g]; yr=fr.kst_dtm.dt.year
        tr=fr[yr.isin(tys)]; va=fr[yr==vy]
        if len(tr)==0 or len(va)==0: continue
        iso=W.fit_powercurve(tr,tgt,cap); tr2=W.with_pc(tr,iso); va2=W.with_pc(va,iso)
        gbm_pred[g]=np.clip(gbm_ens(tr2,va2,V2,tgt),0,cap)
        va_frames[g]=va2
        p=tr2[V2+["kst_dtm"]].copy(); p["cf"]=tr2[tgt]/cap; p["gid"]=g-1; pool.append(p)
    pool=pd.concat(pool,ignore_index=True)
    net,scaler,vmae,ep=train_mlp(pool,V2)
    print(f"fold →{vy}: MLP internal val MAE(cf)={vmae:.4f} (ep {ep}), pool rows={len(pool)}")
    for g in gbm_pred:
        tgt=TGT[g]; cap=W.CAP[g]; va2=va_frames[g]; yt=va2[tgt].to_numpy()
        mp=mlp_predict(net,scaler,va2,V2,g,cap)
        for name,p in [("GBM",gbm_pred[g]),("MLP",mp),
                       ("B75",0.75*gbm_pred[g]+0.25*mp),("B50",0.5*(gbm_pred[g]+mp))]:
            nm,fi=group_scores(yt,np.clip(p,0,cap),cap)
            rows.append(dict(fold=vy,group=g,model=name,nmae=nm,ficr=fi))
cv=pd.DataFrame(rows)
print("\n=== 폴드×그룹 (nmae/ficr) ===")
print(cv.pivot_table(index=["group","fold"],columns="model",values=["nmae","ficr"]).round(4).to_string())

def total(fold,model):
    s=cv[(cv.fold==fold)&(cv.model==model)]
    return 0.5*(1-s.nmae.mean())+0.5*s.ficr.mean()
print("\n=== 공식 총점 ===")
for m in ["GBM","MLP","B75","B50"]:
    print(f"  {m}: 2023={total(2023,m):.4f}  2024={total(2024,m):.4f}")

fold →2023: MLP internal val MAE(cf)=0.1191 (ep 16), pool rows=17328


fold →2024: MLP internal val MAE(cf)=0.1260 (ep 21), pool rows=43602

=== 폴드×그룹 (nmae/ficr) ===
              ficr                            nmae                        
model          B50     B75     GBM     MLP     B50     B75     GBM     MLP
group fold                                                                
1     2023  0.2674  0.2572  0.2493  0.2703  0.1451  0.1461  0.1480  0.1465
      2024  0.3226  0.3369  0.3516  0.3060  0.1234  0.1231  0.1241  0.1276
2     2023  0.3783  0.3795  0.3786  0.3464  0.1345  0.1340  0.1352  0.1405
      2024  0.4215  0.4219  0.4134  0.4070  0.1235  0.1239  0.1254  0.1262
3     2024  0.2620  0.2571  0.2447  0.2635  0.1427  0.1467  0.1525  0.1398

=== 공식 총점 ===
  GBM: 2023=0.5862  2024=0.6013
  MLP: 2023=0.5824  2024=0.5971
  B75: 2023=0.5891  2024=0.6037
  B50: 2023=0.5915  2024=0.6028


## 3. 판정

In [4]:
res={m:(total(2023,m),total(2024,m)) for m in ["GBM","MLP","B75","B50"]}
gbm23,gbm24=res["GBM"]
adopt=None
for m in ["B75","B50"]:
    if res[m][0]>gbm23 and res[m][1]>gbm24:
        if adopt is None or res[m][0]+res[m][1]>res[adopt][0]+res[adopt][1]: adopt=m
verdict=adopt if adopt else "GBM 유지 (blend가 두 해 모두 이기지 못함)"
print("판정:", verdict)
summary=dict(totals={m:{"2023":round(res[m][0],4),"2024":round(res[m][1],4)} for m in res},
             adopted=adopt or "none")
json.dump(summary,open("submission/ver_3/mlp_ablation_summary.json","w"),ensure_ascii=False,indent=2)
print(json.dumps(summary,ensure_ascii=False,indent=2))

판정: B50
{
  "totals": {
    "GBM": {
      "2023": 0.5862,
      "2024": 0.6013
    },
    "MLP": {
      "2023": 0.5824,
      "2024": 0.5971
    },
    "B75": {
      "2023": 0.5891,
      "2024": 0.6037
    },
    "B50": {
      "2023": 0.5915,
      "2024": 0.6028
    }
  },
  "adopted": "B50"
}


## 4. 결론

- 채택 기준(두 해 모두 GBM 단독보다 우위)을 통과한 blend만 최종 파이프라인에 반영.
- 통과 시: `MODELING_V2`의 최종 학습 단계에 MLP를 추가하고 FICR 후처리 재선택 → `submission_v3.csv`.
- 실패 시: GBM 유지 — MLP 다양성 가설 기각(소량·정형에서 흔한 결과). FICR 후처리·group3 개선이 남은 지렛대.